# Solution A1 · Logistic regression with gradient descent

**Type**: Implement from scratch

## Objective

Implement with numpy an L2-regularized logistic regression (loss function, gradient, gradient descent), check the gradient with finite
differences and show that the coefficients match those of `LogisticRegression`. Then show **why** without regularization the method does not converge
on nearly separable data.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)
w0 = np.random.default_rng(0).normal(0, 0.3, Z_tr.shape[1])         # point at which to check the gradient

## Commented solution

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def loss(w, b, Z, y, C):
    n = len(y)
    p = np.clip(sigmoid(Z @ w + b), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)) + w @ w / (2 * C * n)


def gradient(w, b, Z, y, C):
    n = len(y)
    p = sigmoid(Z @ w + b)
    return Z.T @ (p - y) / n + w / (C * n), np.mean(p - y)

**Where `1/(2Cn)` comes from.** scikit-learn minimizes $C\sum_i \ell_i + \tfrac12\lVert w\rVert^2$. Dividing by $Cn$ gives the mean loss
$\frac1n\sum_i\ell_i + \frac{1}{2Cn}\lVert w\rVert^2$, which has the same minimum and is more convenient for the gradient. Large `C` = a penalty that
weighs little.

In [ ]:
gw, gb = gradient(w0, 0.3, Z_tr, y_tr, 1.0)
h = 1e-6
num_w = np.array([(loss(w0 + h * e, 0.3, Z_tr, y_tr, 1.0) - loss(w0 - h * e, 0.3, Z_tr, y_tr, 1.0)) / (2 * h) for e in np.eye(30)])
num_b = (loss(w0, 0.3 + h, Z_tr, y_tr, 1.0) - loss(w0, 0.3 - h, Z_tr, y_tr, 1.0)) / (2 * h)
gradient_error = max(np.abs(gw - num_w).max(), abs(gb - num_b))
print("maximum gap between analytic and numeric gradient:", gradient_error)

In [ ]:
def train(Z, y, C, lr, n_iter):
    w, b = np.zeros(Z.shape[1]), 0.0
    history = []
    for _ in range(n_iter):
        gw, gb = gradient(w, b, Z, y, C)
        w, b = w - lr * gw, b - lr * gb
        history.append(loss(w, b, Z, y, C))
    return w, b, history

In [ ]:
w, b, history = train(Z_tr, y_tr, C=1, lr=0.5, n_iter=5000)
sk = LogisticRegression(C=1, tol=1e-10, max_iter=100000).fit(Z_tr, y_tr)
coef_gap = np.abs(w - sk.coef_[0]).max()
print(f"maximum gap on the coefficients: {coef_gap:.2e} | gap on the intercept: {abs(b - sk.intercept_[0]):.2e}")

In [ ]:
norms = []
for n_iter in (500, 5000):
    w_nr, _, _ = train(Z_tr, y_tr, C=1e6, lr=0.5, n_iter=n_iter)
    norms.append(np.linalg.norm(w_nr))
norms = tuple(norms)
sk_nr = LogisticRegression(C=1e6, tol=1e-10, max_iter=100000).fit(Z_tr, y_tr)
print(f"norm of the weights without regularization: {norms[0]:.2f} after 500 iterations, {norms[1]:.2f} after 5000 | scikit-learn (which stops at the tolerance): {np.linalg.norm(sk_nr.coef_):.1f}")

**Why the norm grows.** On this data the two classes are nearly separable by a hyperplane: the larger the weights, the more the
probabilities approach 0 and 1 on the well-classified examples and the more the log-loss drops. There is no finite minimum: the gradient is always nonzero and the weights
keep growing. L2 regularization adds a term that grows with the norm and **creates** a minimum.

## Check

In [ ]:
with verify():
    check(close(sigmoid(np.array([-50.0, 0.0, 50.0])), [0.0, 0.5, 1.0], atol=1e-12), "sigmoid correct at the extremes (and without overflow)", "sigmoid(-50) = 0, sigmoid(0) = 0.5, sigmoid(50) = 1")
    check(close(loss(np.zeros(30), 0.0, Z_tr, y_tr, 1.0), np.log(2)), "with zero weights the loss is ln 2", "with w = 0 and b = 0 every example costs ln(2)")
    check(gradient_error is not None and gradient_error < 1e-6, "analytic gradient equal to the numeric one (gap < 1e-6)", "check the formula: Z.T @ (p - y) / n + w / (C n) for w, and the mean of (p - y) for b")
    check(len(history) == 5000 and all(np.diff(history) <= 1e-12), "the loss never goes up", "with a small enough step the loss must go down at every iteration")
    check(coef_gap is not None and coef_gap < 1e-3, "the coefficients match those of scikit-learn (gap < 1e-3)", "did you penalize the intercept? used 1/C instead of 1/(C n)?")
    check(norms is not None and norms[1] > 1.5 * norms[0] and norms[1] > 5, "without regularization the norm of the weights keeps growing", "norm of the weights after 5000 iterations against 500, with C = 1e6")
    check(np.all(((Z_te @ w + b) > 0).astype(int) == sk.predict(Z_te)), "the predictions on the test set match those of scikit-learn", "same rule: class 1 if z > 0")
done()

## What you learned

- `fit` is a descent (with a better solver, `lbfgs`) on a convex loss: the mean log-loss plus $\frac{1}{2Cn}\lVert w\rVert^2$. Writing the gradient by hand and
  checking it with finite differences is the standard test for any model you implement: if the gap is 1e-6 or less, the formula is right.
- **`C` is the inverse of the regularization strength** and multiplies the log-loss: the penalty per example is $1/(Cn)$, so with larger datasets
  the same `C` regularizes less.
- The intercept is not penalized: penalizing it would mean depending on the choice of the origin of the variable.
- With separable (or nearly) classes and no penalty, the coefficients diverge: it is a **mathematical fact**, not a defect of the solver. `LogisticRegression` without
  regularization on separable data stops only because the iterations run out or the tolerance is reached. You come back to it with C1.

## Common pitfall

Implementing the logistic gradient with the wrong sign or without dividing by `n`: the loss jumps or does not go down, and you can't tell whether
it is the step's fault or the formula's. The check with finite differences should always be the **first** test.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why the log-loss and not the squared error for logistic regression?</summary>

Because with the sigmoid the squared error is not convex (several local minima) and has tiny gradients when the model is sure and wrong, so it
learns slowly exactly where it matters. The log-loss is the negative log-likelihood of a Bernoulli model (maximum likelihood estimation):
it is convex, its gradient is simply `(p - y) x` and it heavily penalizes whoever is sure and wrong.

</details>